# Target Engineering Quality: $1T Smoke EDA

This gated warehouse notebook measures target construction cost, class support, yearly stability, symbol coverage, and overlap. It does not train a model or select targets using in-sample classifier performance.

In [1]:
from __future__ import annotations
from pathlib import Path
from time import perf_counter
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse.platforms.data_providers.fmp.target_engineering.event_pairs import EventPairStore
from quant_warehouse.research_tools import (
    BinaryTargetConfig, FamilyEvaluationConfig, build_event_target_panel,
    build_oracle_trade_target_panel, combine_target_panels, load_fmp_event_pairs,
    screen_fmp_equity_universe, summarize_binary_targets,
)
from quant_warehouse.warehouse.api import Warehouse

MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '2018-01-01'
END_DATE = None
MIN_POSITIVES_PER_YEAR = 5

def rss_gib() -> float:
    value = next(line for line in Path('/proc/self/status').read_text().splitlines() if line.startswith('VmRSS:')).split()[1]
    return float(value) / 1024 / 1024

print({'python': sys.version.split()[0], 'architecture': os.uname().machine, 'rss_gib': round(rss_gib(), 3)})

{'python': '3.13.11', 'architecture': 'aarch64', 'rss_gib': 0.301}


In [2]:
warehouse = Warehouse()
feature_config = FamilyEvaluationConfig(market_cap_min=MIN_MARKET_CAP, start_date=START_DATE, end_date=END_DATE)
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(feature_config, warehouse=warehouse, required_sections=('prices',))
base_parts = []
for symbol in symbols:
    prices = warehouse.read_prices(symbol, provider='fmp', start=START_DATE, end=END_DATE)
    if prices is None or prices.empty:
        continue
    base_parts.append(pd.DataFrame({'symbol': symbol, 'date': pd.to_datetime(prices.index).normalize()}))
feature_dates = pd.concat(base_parts, ignore_index=True).drop_duplicates(['symbol', 'date'])
print({'universe_source': universe_source, 'symbols': len(symbols), 'symbol_days': len(feature_dates)})

{'universe_source': 'openbb:fmp', 'symbols': 14, 'symbol_days': 27855}


## Build event-pair and oracle-entry targets

In [3]:
target_config = BinaryTargetConfig(
    provider='fmp', start_date=START_DATE, end_date=END_DATE,
    event_families=('congress', 'insider', 'analyst_rating', 'price_target', 'earnings'),
    oracle_trade_k_by_frequency={'YE': (1, 3, 6, 12)},
)
peak_rss = rss_gib()
started = perf_counter()
events, event_diagnostics, event_seconds = load_fmp_event_pairs(
    symbols, target_config, event_store=EventPairStore(backend=warehouse.backend, catalog=warehouse.catalog), include_historical=True
)
event_panel, event_metadata = build_event_target_panel(feature_dates, events, target_config)
peak_rss = max(peak_rss, rss_gib())
oracle_panel, oracle_metadata, oracle_seconds = build_oracle_trade_target_panel(symbols, target_config, warehouse=warehouse)
peak_rss = max(peak_rss, rss_gib())
targets = combine_target_panels(event_panel, oracle_panel)
metadata = pd.concat([event_metadata, oracle_metadata], ignore_index=True).drop_duplicates('target')
summary = summarize_binary_targets(targets, metadata)
print({'events': len(events), 'targets': len(metadata), 'event_seconds': round(event_seconds, 3), 'oracle_seconds': round(oracle_seconds, 3), 'total_seconds': round(perf_counter()-started, 3), 'peak_rss_gib': round(peak_rss, 3)})
display(summary)

{'events': 5167, 'targets': 22, 'event_seconds': 0.862, 'oracle_seconds': 0.946, 'total_seconds': 1.898, 'peak_rss_gib': 0.686}


,target,target_family,rows,rate_rows,positive_rows,positive_rate,positive_symbols,min_positive_date,max_positive_date
0,target_event_on__congress_buy,event,27855,2664,1458,0.547297,9,2018-01-25,2026-06-10
1,target_event_on__congress_sell,event,27855,2664,1431,0.537162,9,2018-01-04,2026-06-16
2,target_event_on__insider_sell,event,27855,538,534,0.992565,7,2018-08-08,2026-06-18
3,target_event_on__analyst_upgrade,event,27855,269,214,0.795539,7,2024-04-04,2026-06-09
4,target_event_on__price_target_raise,event,27855,246,202,0.821138,7,2024-01-11,2026-06-22
5,target_event_on__earnings_beat,event,27855,227,191,0.841410,9,2018-01-31,2026-05-20
6,target_event_on__analyst_downgrade,event,27855,269,69,0.256506,7,2024-04-03,2026-06-25
7,target_event_on__price_target_cut,event,27855,246,61,0.247967,7,2024-01-02,2026-06-25
8,target_event_on__earnings_miss,event,27855,227,36,0.158590,7,2018-02-01,2026-02-05
9,target_event_on__insider_buy,event,27855,538,4,0.007435,2,2021-03-10,2026-02-18


## Annual support, stability, and target overlap

In [4]:
target_columns = [column for column in metadata['target'] if column in targets.columns]
work = targets[['symbol', 'date', *target_columns]].copy()
work['year'] = pd.to_datetime(work['date']).dt.year
annual = work.groupby('year')[target_columns].sum().T.reset_index(names='target')
year_columns = [column for column in annual.columns if isinstance(column, (int, np.integer))]
annual['years_with_support'] = annual[year_columns].ge(MIN_POSITIVES_PER_YEAR).sum(axis=1)
annual['total_positives'] = annual[year_columns].sum(axis=1)
annual = annual.merge(metadata, on='target', how='left').sort_values(['years_with_support', 'total_positives'], ascending=False)
display(annual)

binary = work[target_columns].fillna(0).gt(0).astype('uint8')
active = binary.sum(axis=0)
overlap_rows = []
for left_index, left in enumerate(target_columns):
    for right in target_columns[left_index + 1:]:
        intersection = int((binary[left] & binary[right]).sum())
        union = int((binary[left] | binary[right]).sum())
        if intersection:
            overlap_rows.append({'left': left, 'right': right, 'intersection': intersection, 'jaccard': intersection / union if union else np.nan})
overlap = pd.DataFrame(overlap_rows).sort_values(['jaccard', 'intersection'], ascending=False) if overlap_rows else pd.DataFrame()
display(overlap.head(50))

gate = annual[['target', 'target_family', 'years_with_support', 'total_positives']].copy()
gate['cheap_status'] = np.select(
    [gate['total_positives'].lt(20), gate['years_with_support'].lt(3)],
    ['insufficient_total_support', 'unstable_annual_support'],
    default='candidate_for_wfo',
)
display(gate.groupby(['target_family', 'cheap_status']).size().unstack(fill_value=0))
display(gate.sort_values(['cheap_status', 'total_positives']).head(80))

,target,2018,2019,2020,2021,2022,2023,2024,2025,2026,years_with_support,total_positives,target_family,target_type
21,target_oracle_trade_entry__YE_k12_any,307,301,310,299,311,294,301,307,227,9,2657,oracle_trade,binary
0,target_event_on__congress_buy,75,57,196,114,267,202,155,322,70,9,1458,event,binary
1,target_event_on__congress_sell,39,75,185,155,250,219,190,265,53,9,1431,event,binary
18,target_oracle_trade_entry__YE_k6_any,156,156,156,156,156,156,156,156,160,9,1408,oracle_trade,binary
19,target_oracle_trade_entry__YE_k12_long,156,156,156,155,156,153,152,154,112,9,1350,oracle_trade,binary
20,target_oracle_trade_entry__YE_k12_short,151,145,154,144,155,141,149,153,115,9,1307,oracle_trade,binary
15,target_oracle_trade_entry__YE_k3_any,78,78,78,78,78,78,78,78,82,9,706,oracle_trade,binary
16,target_oracle_trade_entry__YE_k6_long,78,78,78,78,78,78,78,78,80,9,704,oracle_trade,binary
17,target_oracle_trade_entry__YE_k6_short,78,78,78,78,78,78,78,78,80,9,704,oracle_trade,binary
13,target_oracle_trade_entry__YE_k3_long,39,39,39,39,39,39,39,39,41,9,353,oracle_trade,binary


,left,right,intersection,jaccard
170,target_oracle_trade_entry__YE_k6_short,target_oracle_trade_entry__YE_k12_short,704,0.538638
174,target_oracle_trade_entry__YE_k6_any,target_oracle_trade_entry__YE_k12_any,1408,0.529921
167,target_oracle_trade_entry__YE_k6_long,target_oracle_trade_entry__YE_k12_long,704,0.521481
175,target_oracle_trade_entry__YE_k12_long,target_oracle_trade_entry__YE_k12_any,1350,0.508092
58,target_event_on__analyst_upgrade,target_event_on__price_target_raise,139,0.501805
162,target_oracle_trade_entry__YE_k3_any,target_oracle_trade_entry__YE_k6_any,706,0.501420
151,target_oracle_trade_entry__YE_k3_long,target_oracle_trade_entry__YE_k6_long,353,0.501420
156,target_oracle_trade_entry__YE_k3_short,target_oracle_trade_entry__YE_k6_short,353,0.501420
166,target_oracle_trade_entry__YE_k6_long,target_oracle_trade_entry__YE_k6_any,704,0.500000
169,target_oracle_trade_entry__YE_k6_short,target_oracle_trade_entry__YE_k6_any,704,0.500000


cheap_status,candidate_for_wfo,insufficient_total_support,unstable_annual_support
target_family,,,
event,8,1,1
oracle_trade,12,0,0


,target,target_family,years_with_support,total_positives,cheap_status
7,target_event_on__price_target_cut,event,3,61,candidate_for_wfo
5,target_event_on__analyst_downgrade,event,3,69,candidate_for_wfo
10,target_oracle_trade_entry__YE_k1_long,oracle_trade,9,118,candidate_for_wfo
11,target_oracle_trade_entry__YE_k1_short,oracle_trade,9,118,candidate_for_wfo
8,target_event_on__earnings_beat,event,9,191,candidate_for_wfo
6,target_event_on__price_target_raise,event,3,202,candidate_for_wfo
4,target_event_on__analyst_upgrade,event,3,214,candidate_for_wfo
12,target_oracle_trade_entry__YE_k1_any,oracle_trade,9,236,candidate_for_wfo
13,target_oracle_trade_entry__YE_k3_long,oracle_trade,9,353,candidate_for_wfo
14,target_oracle_trade_entry__YE_k3_short,oracle_trade,9,353,candidate_for_wfo


## Gate

Targets with inadequate or unstable label support should not enter the combinatorial model search at larger universes. This is a compute gate, not a claim that sufficiently supported targets are predictively useful. Predictive confirmation belongs in anchored annual WFO in Quant Orchestrator.